# Lab: Modulation and Demodulation of Random Signals

## Objective

Study a sampled simulation of a continuous-time random-signal modulation chain based on white Gaussian noise. The lab has two goals:

1. Analyze the standard modulation chain formed by:
   - a baseband low-pass stage with cutoff frequency $f_n$,
   - multiplication by a cosine carrier $\cos(2\pi f_0 t)$ with $f_0 = k f_n$ and $k = 30$,
   - coherent demodulation followed by low-pass filtering.
2. Compare that coherent receiver with a nonlinear envelope detector made of rectification plus RC smoothing.

The key point is that these two receiver structures do not behave the same way for the modulated random process. The coherent receiver is the reference demodulator for the product-modulated signal, while the envelope detector is included as a comparison case to show what changes when a nonlinear demodulator is used.

At each relevant stage, estimate the autocorrelation functions (ACFs) and power spectral densities (PSDs) of the simulated signals.

## Lab Procedure

1. **Parameter selection**
   - Choose $f_n$ (for example, $f_n = 10$ Hz) and compute $(R_0, C_0)$ such that $f_n = 1/(2\pi R_0 C_0)$.
   - Set $f_0 = k f_n$ with $k = 30$ (for example, $f_0 = 300$ Hz when $f_n = 10$ Hz).
   - Choose a sampling frequency $f_s \gg 2 f_0$ and a sufficiently long observation time $T$ to estimate ACFs and PSDs with good resolution.
   - Choose coherent-demodulation filter parameters so the post-multiplier low-pass stage preserves the baseband bandwidth around $f_n$.
   - Choose envelope-detector smoothing parameters so the RC stage suppresses carrier ripple while tracking the baseband envelope, for example with a cutoff frequency $f_{env}$ slightly above $f_n$.

2. **Stage 0: Generation of the input signal**
   - Generate discrete-time white Gaussian noise $x[n]$ with zero mean and unit variance.
   - Estimate $R_{xx}[\ell]$ using sample autocorrelation.
   - Estimate $S_{xx}(f)$ using the FFT of $x[n]$ or a periodogram method.
   - Plot the ACF and PSD of $x[n]$.

3. **Stage 1: Baseband low-pass filtering**
   - Implement the RC low-pass $(R_0, C_0)$ as a digital filter, for example a first-order IIR with cutoff $f_n$, and obtain the band-limited process $x_1[n]$.
   - Estimate $R_{x_1 x_1}[\ell]$ using sample autocorrelation.
   - Estimate $S_{x_1 x_1}(f)$ using a periodogram method.
   - Plot the ACF and PSD of $x_1[n]$.

4. **Stage 2: Cosine modulation**
   - Form the modulated signal $x_2[n] = x_1[n]\cos(2\pi f_0 n T_s)$, with $T_s = 1/f_s$.
   - Estimate $R_{x_2 x_2}[\ell]$ and $S_{x_2 x_2}(f)$.
   - Observe that the ACF of $x_2[n]$ exhibits an oscillatory factor at $f_0$ and that the PSD shows translated sidebands centered at $\pm f_0$.

5. **Stage 3A: Coherent demodulation reference path**
   - Multiply the modulated signal again by the synchronized carrier to form $x_{c}[n] = x_2[n]\cos(2\pi f_0 n T_s)$.
   - Apply a low-pass filter to obtain the coherent-receiver output $x_{coh}[n]$.
   - Estimate $R_{x_{coh} x_{coh}}[\ell]$ and $S_{x_{coh} x_{coh}}(f)$.
   - Compare $x_{coh}[n]$ with $x_1[n]$ and identify the expected scale factor introduced by modulation and demodulation.

6. **Stage 3B: Envelope-detector comparison path**
   - Rectify the modulated signal $x_2[n]$.
   - Apply an RC smoothing filter to obtain the envelope-detector output $x_{env}[n]$.
   - Estimate $R_{x_{env} x_{env}}[\ell]$ and $S_{x_{env} x_{env}}(f)$.
   - Compare $x_{env}[n]$ with $x_1[n]$ and describe which features are preserved and which are lost due to nonlinear detection.

7. **Comparison and discussion**
   - Plot on common axes representative ACFs such as $R_{x_1 x_1}(\tau)$, $R_{x_2 x_2}(\tau)$, $R_{x_{coh} x_{coh}}(\tau)$, and $R_{x_{env} x_{env}}(\tau)$.
   - Plot on common axes the corresponding PSDs $S_{x_1 x_1}(f)$, $S_{x_2 x_2}(f)$, $S_{x_{coh} x_{coh}}(f)$, and $S_{x_{env} x_{env}}(f)$.
   - Explain why coherent demodulation is the appropriate recovery method for the product-modulated signal.
   - Explain why the envelope detector does not reproduce the original baseband process exactly, even when its output still reflects the slow envelope variation.
   - Relate the observed changes in ACF and PSD to the theoretical effects of linear filtering, modulation, synchronous detection, and nonlinear rectification.

## Simulation Model

Use the following discrete-time model throughout the lab.

Let $T_s = 1/f_s$ and define the one-pole RC coefficient

$$
\alpha(f_c) = e^{-2\pi f_c T_s}.
$$

For any RC low-pass stage with cutoff frequency $f_c$, implement the recursion

$$
y[n] = (1-\alpha(f_c))\,x[n] + \alpha(f_c)\,y[n-1],
$$

with initial condition $y[0] = (1-\alpha(f_c))x[0]$.

Using that convention, the complete simulation chain is:

1. **White-noise source**
   - Generate $x[n] \sim \mathcal{N}(0,1)$ independently.

2. **Baseband low-pass stage**
   - Use cutoff $f_n$.
   - Compute
$$
x_1[n] = (1-\alpha_n)x[n] + \alpha_n x_1[n-1], \qquad \alpha_n = \alpha(f_n).
$$

3. **Carrier modulation**
   - Define the carrier
$$
c[n] = \cos(2\pi f_0 n T_s).
$$
   - Form
$$
x_2[n] = x_1[n]c[n].
$$

4. **Coherent receiver branch**
   - Multiply again by the synchronized carrier:
$$
x_c[n] = x_2[n]c[n].
$$
   - Apply the same low-pass structure with cutoff $f_n$:
$$
x_{coh}[n] = (1-\alpha_n)x_c[n] + \alpha_n x_{coh}[n-1].
$$

5. **Envelope-detector branch**
   - Use **full-wave rectification**:
$$
x_r[n] = |x_2[n]|.
$$
   - Smooth with another RC low-pass stage with cutoff $f_{env}$:
$$
x_{env}[n] = (1-\alpha_{env})x_r[n] + \alpha_{env}x_{env}[n-1], \qquad \alpha_{env} = \alpha(f_{env}).
$$

This notebook therefore studies one linear reference receiver and one nonlinear comparison receiver under the same input and carrier conditions.

## Default Parameters

Use these defaults unless the instructor specifies alternatives:

- Baseband cutoff: $f_n = 10$ Hz.
- Carrier ratio: $k = 30$.
- Carrier frequency: $f_0 = k f_n = 300$ Hz.
- Sampling frequency: $f_s = 5000$ Hz.
- Observation time: $T = 20$ s.
- Number of samples: $N = T f_s = 100000$.
- Envelope-detector smoothing cutoff: $f_{env} = 15$ Hz.
- Random seed: use a fixed seed such as `12345` for reproducible plots.

These values ensure that:

- the carrier is well above the baseband bandwidth,
- the sampling rate comfortably exceeds $2f_0$,
- the record length is long enough to estimate smooth ACFs and PSDs,
- the envelope smoother tracks slow baseband variation while attenuating carrier ripple.

## Estimation Conventions

Use one consistent set of estimators across all stages.

1. **Sample ACF**
   - Remove the sample mean before estimating the ACF unless you are explicitly studying the detector DC term.
   - Use the biased estimator
$$
\hat R_{xx}[\ell] = \frac{1}{N}\sum_{n=0}^{N-1-|\ell|} \tilde x[n]\tilde x[n+|\ell|],
$$
   where $\tilde x[n] = x[n] - \bar x$.
   - Plot lags over a symmetric window such as $|\ell| \leq 2000$.

2. **PSD estimate**
   - Use a periodogram or Welch estimate, but keep the same method for every signal.
   - Prefer a two-sided PSD centered at zero frequency using FFT shifting, since modulation creates components at both positive and negative frequencies.
   - State clearly in the notebook which estimator you used.

3. **Amplitude comparisons**
   - When comparing $x_{coh}[n]$ with $x_1[n]$, allow for the expected scale factor from synchronous demodulation.
   - When comparing $x_{env}[n]$ with $x_1[n]$, compare shape and second-order behavior, not sign-by-sign waveform equality.

## Expected Results

Your simulations should support the following conclusions.

1. **Input white noise $x[n]$**
   - The ACF is strongly concentrated near zero lag.
   - The PSD is approximately flat over the discrete-time frequency range, up to finite-record variation.

2. **Baseband process $x_1[n]$**
   - The ACF becomes wider and smoother than that of white noise.
   - The PSD becomes low-pass and concentrated near zero frequency.

3. **Modulated process $x_2[n]$**
   - The ACF acquires an oscillatory factor at the carrier frequency.
   - The PSD shifts from baseband to two sidebands centered near $\pm f_0$.

4. **Coherent receiver output $x_{coh}[n]$**
   - After low-pass filtering, the signal returns to baseband.
   - Its waveform should match a scaled version of $x_1[n]$.
   - Its ACF and PSD should have the same shape as those of $x_1[n]$, up to a scale factor.
   - For ideal synchronous demodulation, the main expected scale is approximately $1/2$ in amplitude after the second multiplication and low-pass recovery.

5. **Envelope-detector output $x_{env}[n]$**
   - The output follows slow envelope variation but does not preserve the sign of $x_1[n]$.
   - It generally has a positive mean due to rectification.
   - Its ACF and PSD differ from those of $x_1[n]$ because rectification is nonlinear.
   - It should not be interpreted as exact recovery of the original product-modulated baseband process.

## Deliverables

A complete lab submission should include the following.

1. **Parameter table**
   - Report the values of $f_n$, $f_0$, $f_s$, $T$, $N$, $f_{env}$, and the random seed.

2. **Implementation summary**
   - State the exact recursion used for the RC filters.
   - State whether the PSD was computed with a periodogram or Welch estimate.
   - State whether the ACF was mean-removed and biased or unbiased.

3. **Required plots**
   - Time realizations of $x[n]$, $x_1[n]$, $x_2[n]$, $x_{coh}[n]$, and $x_{env}[n]$ over a short interval.
   - ACF plots for $x[n]$, $x_1[n]$, $x_2[n]$, $x_{coh}[n]$, and $x_{env}[n]$.
   - PSD plots for the same signals on consistent axes.
   - At least one direct comparison plot of $x_1[n]$ and the appropriately scaled $x_{coh}[n]$.

4. **Discussion questions**
   - Why does modulation shift the PSD away from zero frequency?
   - Why does coherent demodulation recover the baseband shape up to a scale factor?
   - Why does envelope detection fail to reproduce the signed baseband process exactly?
   - How do the observed ACFs support the theoretical interpretation of each stage?

5. **Short conclusion**
   - Summarize which receiver preserves the original second-order structure more faithfully.
   - State whether the envelope detector is appropriate for this product-modulated random signal and justify the answer from the plots and statistics.
